In [1]:
import httpx
from worker.database import connect

In [2]:
from worker.settings import Settings


ROOT_URL = "https://api.marketstack.com/v2"
EOD_URL = f"{ROOT_URL}/eod"

settings = Settings()

In [4]:
from sqlalchemy import select
import polars as pl

from worker.database import Instrument, MarketData


with connect() as session:
    insts = session.scalars(select(Instrument)).all()
    symbols = [inst.ticker for inst in insts]
    df_insts = pl.DataFrame(
        [
            {
                "id": str(inst.id),
                "ticker": inst.ticker,
            }
            for inst in insts
        ]
    )

    market_data = session.scalars(select(MarketData)).all()

    df_market_data = pl.DataFrame(
        [
            {
                "instrument_id": str(item.instrument_id),
                "date": item.date,
                "data_type": item.data_type,
                "value": item.value,
            }
            for item in market_data
        ],
        schema={
            "instrument_id": pl.String,
            "date": pl.Date,
            "data_type": pl.String,
            "value": pl.Float64,
        },
    )


In [5]:
df_market_data

instrument_id,date,data_type,value
str,date,str,f64
"""559e6146-595c-4ea6-a489-0415f4…",2025-08-28,"""open""",49.01
"""929e3e08-574e-4b80-ab1d-57bdef…",2025-08-28,"""open""",66.4
"""3c7f3c30-4848-48ce-b2a5-3fc1b9…",2025-08-28,"""open""",37.68
"""70e9623b-1ea8-4de7-a8eb-4193d4…",2025-08-28,"""open""",18.19
"""588cf2ec-8e51-4360-8d55-8c807a…",2025-08-28,"""open""",99.43
…,…,…,…
"""41712141-41c4-4e22-83d1-8cc45c…",2025-08-28,"""open""",49.86
"""b784bce9-a223-49a4-9c09-9f381c…",2025-08-28,"""open""",92.0
"""29fcfd80-cc50-483f-b2f1-a70992…",2025-08-28,"""open""",62.98


In [ ]:
from worker.data_loader import _build_ticker_lists
batches = _build_ticker_lists(symbols, max_tickers=100)

In [7]:
len(batches)

1

In [13]:
# Download 10 years of data
offset = 0

async with httpx.AsyncClient() as client:
    while True:
        print(f"Processing {offset=}")
        resp = await client.get(
            EOD_URL,
            params={
                "access_key": settings.MARKETSTACK_API_KEY,
                "symbols": batches[0],
                "date_from": "2015-09-01",
                "offset": offset,
                "limit": 1000,
            },
        )

        json = resp.json()
        pagination = json["pagination"]

        df = (
            pl.DataFrame(json["data"])
            .drop("name", "exchange_code", "asset_type", "price_currency", "exchange")
            .with_columns(date=pl.col("date").str.to_datetime("%Y-%m-%dT%H:%M:%S+%Z").dt.date())
            .unpivot(index=["date", "symbol"])
            .rename({"variable": "data_type"})
        )

        df_with_ids = (
            df.join(df_insts, left_on="symbol", right_on="ticker")
            .rename(
                {
                    "id": "instrument_id",
                }
            )
            .drop("symbol")
        )

        df_unique = (
            df_with_ids.join(df_market_data, how="left", on=["date", "instrument_id", "data_type"])
            .filter(pl.col("value_right").is_null())
            .drop("value_right")
        )

        items_to_add = df_unique.to_dicts()

        from sqlalchemy.dialects.postgresql import insert

        stmt = insert(MarketData).values(items_to_add)
        stmt = stmt.on_conflict_do_nothing(index_elements=['instrument_id', 'date', 'data_type'])
        session.execute(stmt)
        session.commit()
        
        offset += 1000

        if len(df) < 1000 or offset > pagination["total"]:
            break


Processing offset=0


PendingRollbackError: This Session's transaction has been rolled back due to a previous exception during flush. To begin a new transaction with this Session, first issue Session.rollback(). Original exception was: (psycopg2.errors.UniqueViolation) duplicate key value violates unique constraint "pkey"
DETAIL:  Key (instrument_id, date, data_type)=(3c7f3c30-4848-48ce-b2a5-3fc1b9e31b1e, 2025-08-28, high) already exists.

[SQL: INSERT INTO market_data (date, instrument_id, data_type, value) VALUES (%(date__0)s, %(instrument_id__0)s::UUID, %(data_type__0)s, %(value__0)s), (%(date__1)s, %(instrument_id__1)s::UUID, %(data_type__1)s, %(value__1)s), (%(date__2)s, %(instrument_id ... 82279 characters truncated ... , %(value__998)s), (%(date__999)s, %(instrument_id__999)s::UUID, %(data_type__999)s, %(value__999)s)]
[parameters: {'value__0': 86.78, 'instrument_id__0': '6c125ed4-4a90-41c4-b389-226eb7e93b07', 'data_type__0': 'open', 'date__0': datetime.date(2025, 8, 28), 'value__1': 111.24, 'instrum

date,data_type,instrument_id
date,str,str
2025-08-12,"""split_factor""","""fe4ea713-6883-489f-bd81-8b3e1e…"
2025-08-07,"""dividend""","""6c125ed4-4a90-41c4-b389-226eb7…"
2025-08-12,"""dividend""","""50d64c2b-1484-41db-8095-d18e20…"
2025-08-04,"""adj_volume""","""559e6146-595c-4ea6-a489-0415f4…"
2025-08-12,"""adj_close""","""f48b1e9f-e2d6-442a-acf6-2ba362…"
…,…,…
2025-08-19,"""adj_high""","""b943b709-b920-44a1-a0c2-c056b6…"
2025-08-06,"""adj_open""","""1fa0820d-a943-4d8a-8394-adafc9…"
2025-08-11,"""adj_high""","""727b0b7c-268d-42c4-bb48-11020f…"


In [ ]:
pagination

{'limit': 10, 'offset': 0, 'count': 10, 'total': 122444}

In [11]:
import polars as pl

df = (
    pl.DataFrame(json["data"])
    .drop("name", "exchange_code", "asset_type", "price_currency", "exchange")
    .with_columns(date=pl.col("date").str.to_datetime("%Y-%m-%dT%H:%M:%S+%Z").dt.date())
    .unpivot(index=["date", "symbol"])
    .rename({"variable": "data_type"})
)

In [ ]:
df_with_ids = (
    df.join(df_insts, left_on="symbol", right_on="ticker")
    .rename(
        {
            "id": "instrument_id",
        }
    )
    .drop("symbol")
)

df_unique = (
    df_with_ids.join(df_market_data, how="left", on=["date", "instrument_id", "data_type"])
    .filter(pl.col("value_right").is_null())
    .drop("value_right")
)

In [32]:
items_to_add = df_unique.to_dicts()

with connect() as session:
    for item in items_to_add:
        obj = MarketData(**item)
        session.add(obj)

    session.commit()


/var/folders/93/m5rz0kz50ys3_8j8kz62d1b00000gn/T/ipykernel_51361/3435652783.py:8: SAWarning: Identity map already had an identity for (<class 'worker.database.MarketData'>, (datetime.date(2025, 8, 28), '588cf2ec-8e51-4360-8d55-8c807abd11c8'), None), replacing it with newly flushed object.   Are there load operations occurring inside of an event handler within the flush?
  session.commit()
/var/folders/93/m5rz0kz50ys3_8j8kz62d1b00000gn/T/ipykernel_51361/3435652783.py:8: SAWarning: Identity map already had an identity for (<class 'worker.database.MarketData'>, (datetime.date(2025, 8, 28), '29fcfd80-cc50-483f-b2f1-a7099277e15a'), None), replacing it with newly flushed object.   Are there load operations occurring inside of an event handler within the flush?
  session.commit()
/var/folders/93/m5rz0kz50ys3_8j8kz62d1b00000gn/T/ipykernel_51361/3435652783.py:8: SAWarning: Identity map already had an identity for (<class 'worker.database.MarketData'>, (datetime.date(2025, 8, 28), '41712141-41c4

In [17]:
df_market_data

shape: (0, 0)
┌┐
╞╡
└┘